# Step 5: Fine-Tuning the COCO-Pretrained EoMT on Cityscapes

The COCO-pretrained model covers 133 classes across a broad range of everyday
scenes, but it has not been exposed to the specific distribution of road imagery
in Cityscapes. The goal of this step is to adapt the model to the 19-class
Cityscapes semantic segmentation task and measure how performance changes after
fine-tuning.

We run three experiments with increasing degrees of backbone adaptation:
  1. Head-only: the backbone is fully frozen, only the prediction head is trained.
  2. Gradual unfreeze: the last transformer block is also unfrozen.
  3. LoRA: low-rank adapters are injected into the attention layers.

All experiments share the same optimizer and scheduler so that differences
in validation mIoU are attributable to the freezing strategy alone.

## 1. Environment Setup

In [1]:
# Install dependencies (output suppressed for readability)
!pip install lightning > /dev/null
!pip install gitignore_parser > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' > /dev/null
!pip install -U "torchao>=0.16.0" > /dev/null
!pip install peft > /dev/null
!pip install wandb > /dev/null

In [2]:
from google.colab import drive, userdata
import os, sys, json, yaml, types, glob, shutil
import torch
import torch.nn.functional as F
import wandb
from tqdm import tqdm
from lightning import seed_everything
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import WandbLogger
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from peft import LoraConfig, get_peft_model

In [3]:
# PyTorch 2.6 changed weights_only default to True, which blocks custom classes
# in Lightning checkpoints. Safe to patch here since these are our own files.
_original_torch_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_load

In [4]:
# Mount Drive and configure paths
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

project_root = '/content/drive/MyDrive/FundGitHubProject'
if not os.path.exists('/content/ProjectFolder'):
    !ln -s /content/drive/MyDrive/FundGitHubProject /content/ProjectFolder

eomt_folder = project_root + '/eomt'
os.chdir(project_root)
for p in [project_root, eomt_folder]:
    if p not in sys.path:
        sys.path.insert(0, p)

Mounted at /content/drive


In [5]:
from eval.iouEval import iouEval
from training.mask_classification_panoptic import MaskClassificationPanoptic
from training.mask_classification_semantic import MaskClassificationSemantic
from models.eomt import EoMT
from models.vit import ViT
from datasets.cityscapes_semantic import CityscapesSemantic

In [6]:
seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [7]:
# %%
# Copy the dataset from Drive to local Colab storage.
# This is a one-time cost per session (~15 min) but avoids I/O bottlenecks
# during training. All subsequent cells use the local path.

drive_data_path = os.path.join(eomt_folder, 'data')
local_data_path = '/content/cityscapes_data'

if not os.path.exists(local_data_path):
    print("Copying dataset to local storage, this takes a few minutes...")
    shutil.copytree(drive_data_path, local_data_path)
    print("Done.")

data_path = local_data_path

Copying dataset to local storage, this takes a few minutes...
Done.


## 2. Zero-Shot Baseline (The Step 4 Score)

Before any fine-tuning, we replicate the Step 4 evaluation on the Cityscapes
validation set. This confirms that the evaluation pipeline is consistent and
provides a reference point for measuring the benefit of fine-tuning.

The COCO model produces panoptic predictions over 133 classes. We map these to
Cityscapes train IDs using the same thing/stuff mapping from Step 4.

In [8]:
coco_cfg_path = os.path.join(eomt_folder, 'configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml')
with open(coco_cfg_path, "r") as f:
    coco_config = yaml.safe_load(f)

bin_path = os.path.join(eomt_folder, 'eomt_weights/eomt_coco.bin')

encoder = ViT(img_size=640, backbone_name="vit_base_patch14_reg4_dinov2")
network = EoMT(num_classes=133, encoder=encoder, num_q=200, num_blocks=3, masked_attn_enabled=False)
model_coco = MaskClassificationPanoptic(
    network=network,
    img_size=(640, 640),
    num_classes=133,
    stuff_classes=coco_config["data"].get("init_args", {}).get("stuff_classes", []),
    attn_mask_annealing_enabled=False
)
ckpt = torch.load(bin_path, map_location="cpu")
model_coco.load_state_dict(ckpt.get("state_dict", ckpt), strict=False)
model_coco.to(device).eval()
print("COCO model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


COCO model loaded.


In [9]:
# Class mapping: COCO 80-class indices -> COCO 91-class IDs -> Cityscapes train IDs
# Identical to Step 4 to ensure evaluation consistency.
map_file = os.path.join(project_root, 'coco-classes-mapping-master/coco_mapping_80to91.json')
with open(map_file, 'r') as f:
    coco_idx_map = {int(k)-1: int(v) for k, v in json.load(f).items()}

things_map = {1: 11, 2: 18, 3: 13, 4: 17, 6: 15, 7: 16, 8: 14, 10: 6, 13: 7}
stuff_map  = {100: 0, 123: 1, 91: 2, 129: 2, 109: 3, 110: 3, 111: 3,
              112: 3, 131: 3, 117: 4, 116: 8, 125: 8, 126: 9, 119: 10}

def bridge_to_cs(pred_tensor):
    """Map COCO panoptic predictions to Cityscapes train IDs (19 = ignore)."""
    res = torch.full_like(pred_tensor, 19)
    for idx, coco_id in coco_idx_map.items():
        if coco_id in things_map:
            res[pred_tensor == idx] = things_map[coco_id]
    for stuff_id, cs_id in stuff_map.items():
        res[pred_tensor == stuff_id] = cs_id
    return res

In [10]:
## Batch size

dm_zeroshot = CityscapesSemantic(path=data_path, batch_size=1, num_workers=2)
dm_zeroshot.setup()

evaluator = iouEval(20)
for batch in tqdm(dm_zeroshot.val_dataloader(), desc="Zero-shot eval"):
    imgs, targets = batch
    gt = model_coco.to_per_pixel_targets_semantic(targets, 19)[0].to(device)
    with torch.no_grad():
        tx = model_coco.resize_and_pad_imgs_instance_panoptic([imgs[0].to(device)])
        mp, cp = model_coco(tx)
        mp = model_coco.revert_resize_and_pad_logits_instance_panoptic(
            F.interpolate(mp[-1], model_coco.img_size, mode="bilinear"),
            [imgs[0].shape[-2:]]
        )
        pred = model_coco.to_per_pixel_preds_panoptic(
            mp, cp[-1], model_coco.stuff_classes, 0.8, 0.8
        )[0][..., 0]
        evaluator.addBatch(
            bridge_to_cs(pred).unsqueeze(0).unsqueeze(0),
            gt.unsqueeze(0).unsqueeze(0)
        )

_, ious = evaluator.getIoU()
zeroshot_miou = ious[:19].mean() * 100
print(f"Zero-shot mIoU: {zeroshot_miou:.2f}%")

KeyboardInterrupt: 

## 3. Fine-Tuning Setup

### Model initialization

The COCO model uses a 133-class panoptic head. For fine-tuning on Cityscapes we
load the same backbone and transformer weights, but replace the head with a fresh
19-class semantic head. The class MaskClassificationSemantic handles this when
load_ckpt_class_head=False is set.

We keep the image resolution at 640x640 to match the positional embeddings in the
pretrained ViT backbone. The checkpoint encodes 40x40 patch tokens (640/16=40);
using a different resolution would require interpolating those embeddings.

### Shared training configuration

All three experiments use:
- AdamW with lr=2e-4 and weight_decay=0.01
- Cosine annealing LR scheduler over the full training run
- AMP (16-bit mixed precision) to reduce GPU memory usage on Colab
- batch_size=4, img_size=(640, 640)

Keeping these identical means any difference in validation mIoU between
experiments is attributable solely to the freezing strategy.

In [11]:
def build_finetune_model():
    """
    Initialize EoMT with COCO backbone weights and a fresh 19-class head.
    Called at the start of each experiment to ensure a clean starting point.
    """
    encoder_ft = ViT(img_size=640, backbone_name="vit_base_patch14_reg4_dinov2")
    network_ft = EoMT(
        num_classes=19,
        encoder=encoder_ft,
        num_q=200,
        num_blocks=3,
        masked_attn_enabled=True
    )
    model = MaskClassificationSemantic(
        network=network_ft,
        img_size=(640, 640),
        num_classes=19,
        load_ckpt_class_head=False,
        ckpt_path=bin_path,
        attn_mask_annealing_enabled=True
    )
    model.attn_mask_annealing_enabled = False
    # The built-in image logging in the model is incompatible with this setup
    model.plot_semantic = types.MethodType(lambda *args, **kwargs: None, model)
    return model


def configure_optimizer(self):
    """AdamW + cosine annealing, operating only on parameters that require grad."""
    trainable_params = [p for p in self.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=2e-4, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=self.trainer.max_epochs)
    return {
        "optimizer": optimizer,
        "lr_scheduler": {"scheduler": scheduler, "interval": "epoch", "frequency": 1}
    }


def build_trainer(run_name, max_epochs=10):
    """Set up the Lightning Trainer with WandB logging and checkpointing."""
    wandb.finish()
    wandb.login(key=userdata.get('WANDDB-API-KEY'))

    wandb_logger = WandbLogger(
        project="eomt-cityscapes-finetuning",
        name=run_name
    )
    checkpoint_cb = ModelCheckpoint(
        dirpath=os.path.join(project_root, 'checkpoints', run_name),
        filename='eomt-{epoch:02d}',
        save_top_k=2,
        monitor='metrics/val_iou_all',
        mode='max',
        save_last=True,      # always write last.ckpt so we can resume after a crash
        every_n_epochs=1
    )
    trainer = Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=1,
        callbacks=[LearningRateMonitor(logging_interval='step'), checkpoint_cb],
        logger=wandb_logger,
        precision="16-mixed",
        log_every_n_steps=10
    )
    trainer.num_sanity_val_steps = 0
    return trainer


def find_latest_ckpt(run_name):
    """
    Returns the path to the most recent checkpoint for a given run,
    or None if no checkpoint exists yet (first run).
    Prefers last.ckpt over best checkpoints since it is the most recent epoch.
    """
    ckpt_dir = os.path.join(project_root, 'checkpoints', run_name)
    last = os.path.join(ckpt_dir, 'last.ckpt')
    if os.path.exists(last):
        print(f"Resuming from: {last}")
        return last
    ckpts = sorted(glob.glob(os.path.join(ckpt_dir, '*.ckpt')))
    if ckpts:
        print(f"Resuming from: {ckpts[-1]}")
        return ckpts[-1]
    print(f"No checkpoint found for '{run_name}', starting from scratch.")
    return None

In [12]:
# DataModule used for all three training experiments
dm_train = CityscapesSemantic(
    path=data_path,
    batch_size=4,
    num_workers=2,
    img_size=(640, 640)
)
dm_train.setup("fit")

In [ ]:
"""# Quick test: 3 train batches + 3 val batches, takes ~2 minutes
trainer_test = Trainer(
    max_epochs=1,
    limit_train_batches=3,
    limit_val_batches=3,
    accelerator="auto",
    devices=1,
    precision="16-mixed"
)
trainer_test.num_sanity_val_steps = 0
test_model = build_finetune_model()
test_model.configure_optimizers = types.MethodType(configure_optimizer, test_model)
trainer_test.fit(model=test_model, datamodule=dm_train)

# This is what matters — print everything logged
print(trainer_test.callback_metrics)"""

'# Quick test: 3 train batches + 3 val batches, takes ~2 minutes\ntrainer_test = Trainer(\n    max_epochs=1,\n    limit_train_batches=3,\n    limit_val_batches=3,\n    accelerator="auto",\n    devices=1,\n    precision="16-mixed"\n)\ntrainer_test.num_sanity_val_steps = 0\ntest_model = build_finetune_model()\ntest_model.configure_optimizers = types.MethodType(configure_optimizer, test_model)\ntrainer_test.fit(model=test_model, datamodule=dm_train)\n\n# This is what matters — print everything logged\nprint(trainer_test.callback_metrics)'

In [ ]:
"""import inspect
print(inspect.getsource(MaskClassificationSemantic.validation_step))"""

'import inspect\nprint(inspect.getsource(MaskClassificationSemantic.validation_step))'

In [ ]:
#print(inspect.getsource(MaskClassificationSemantic.on_validation_epoch_end))

## 4. Experiment 1: Head-Only Fine-Tuning

The backbone and transformer blocks are fully frozen. Only the 19-class prediction
head (randomly initialized) is trained. This is the most conservative strategy and
the natural first experiment: it tells us how much of the performance gap between
zero-shot and full supervision can be closed by simply re-learning the classifier
on top of the fixed COCO features.

Since nothing in the encoder is updated, there is no risk of catastrophic
forgetting of the pretrained representations.

In [13]:
def configure_optimizer_exp1(self):
    """
    Head-only: higher LR is safe since the backbone is frozen.
    Gradient clipping is handled at the Trainer level (gradient_clip_val=1.0).
    """
    trainable_params = [p for p in self.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=5e-4, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=self.trainer.max_epochs)
    return {
        "optimizer": optimizer,
        "lr_scheduler": {"scheduler": scheduler, "interval": "epoch", "frequency": 1}
    }

In [14]:
def build_trainer_exp1(run_name, max_epochs=10):
    wandb.finish()
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    wandb_logger = WandbLogger(
        project="eomt-cityscapes-finetuning",
        name=run_name
    )
    checkpoint_cb = ModelCheckpoint(
        dirpath=os.path.join(project_root, 'checkpoints', run_name),
        filename='eomt-{epoch:02d}',
        save_top_k=2,
        monitor='metrics/val_iou_all',
        mode='max',
        save_last=True,
        every_n_epochs=1
    )
    trainer = Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=1,
        callbacks=[LearningRateMonitor(logging_interval='step'), checkpoint_cb],
        logger=wandb_logger,
        precision="16-mixed",
        log_every_n_steps=10,
        gradient_clip_val=1.0,     # stabilizes head training, standard in transformers
        accumulate_grad_batches=4   # effective batch_size = 4 * 4 = 16
    )
    trainer.num_sanity_val_steps = 0
    return trainer

In [ ]:
model_exp1 = build_finetune_model()

for param in model_exp1.parameters():
    param.requires_grad = False
for param in model_exp1.network.class_head.parameters():
    param.requires_grad = True

n = sum(p.numel() for p in model_exp1.parameters() if p.requires_grad)
print(f"Trainable parameters: {n:,}")

model_exp1.configure_optimizers = types.MethodType(configure_optimizer_exp1, model_exp1)
model_exp1.train()

trainer_exp1 = build_trainer_exp1("head-only-new", max_epochs=10)
trainer_exp1.fit(
    model=model_exp1,
    datamodule=dm_train,
    ckpt_path=find_latest_ckpt("head-only-new")
)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


Trainable parameters: 15,380


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightni

No checkpoint found for 'head-only-new', starting from scratch.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network   │ EoMT                   │ 93.6 M │ train │     0 │
│ 1 │ criterion │ MaskClassificationLoss │      0 │ train │     0 │
│ 2 │ metrics   │ ModuleList             │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 15.4 K                                                                                           
Non-trainable params: 93.6 M                                                                                       
Total params: 93.6 M                                                                                               
Total estimated model params size (MB): 374.302                                                                    
Modules in train mode: 304                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

INFO: mIoU: 60.9
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 60.9
INFO: mIoU: 63.3
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 63.3
INFO: mIoU: 63.3
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 63.3
INFO: mIoU: 63.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 63.6


## 5. Experiment 2: Gradual Unfreezing

In this experiment we additionally unfreeze the last transformer block of the ViT
encoder. The motivation is that the deeper layers of a vision transformer encode
increasingly task-specific, high-level representations, while the shallower layers
capture low-level features (edges, textures) that transfer well across domains.
Allowing the last block to adapt gives the model flexibility to adjust its
top-level features to Cityscapes without modifying the more universal early layers.

This is a middle ground between head-only training and full fine-tuning, and is
a common strategy when compute is limited.

In [ ]:
from torch.optim.lr_scheduler import SequentialLR, LinearLR

def configure_optimizer_exp2(self):
    """
    LLRD: head gets lr=2e-4, backbone block gets lr=2e-5 (10x lower).
    Linear warmup over 3 epochs, then cosine decay for the remaining epochs.
    """
    head_params     = list(self.network.class_head.parameters())
    backbone_params = list(
        self.network.encoder.backbone.blocks[-1].parameters()
    )

    optimizer = AdamW([
        {"params": head_params,     "lr": 2e-4},   # randomly initialized
        {"params": backbone_params, "lr": 2e-5},   # pretrained, update gently
    ], weight_decay=0.01)

    warmup_epochs = 3
    warmup  = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                       total_iters=warmup_epochs)
    cosine  = CosineAnnealingLR(optimizer,
                                T_max=self.trainer.max_epochs - warmup_epochs)
    scheduler = SequentialLR(optimizer,
                             schedulers=[warmup, cosine],
                             milestones=[warmup_epochs])

    return {
        "optimizer": optimizer,
        "lr_scheduler": {"scheduler": scheduler, "interval": "epoch", "frequency": 1}
    }


def build_trainer_exp2(run_name, max_epochs=10):
    wandb.finish()
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    wandb_logger = WandbLogger(
        project="eomt-cityscapes-finetuning",
        name=run_name
    )
    checkpoint_cb = ModelCheckpoint(
        dirpath=os.path.join(project_root, 'checkpoints', run_name),
        filename='eomt-{epoch:02d}',
        save_top_k=2,
        monitor='metrics/val_iou_all',
        mode='max',
        save_last=True,
        every_n_epochs=1
    )
    trainer = Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        devices=1,
        callbacks=[LearningRateMonitor(logging_interval='step'), checkpoint_cb],
        logger=wandb_logger,
        precision="16-mixed",
        log_every_n_steps=10,
        gradient_clip_val=1.0     # protects backbone from large head gradients
    )
    trainer.num_sanity_val_steps = 0
    return trainer

In [ ]:
model_exp2 = build_finetune_model()

for param in model_exp2.parameters():
    param.requires_grad = False
for param in model_exp2.network.class_head.parameters():
    param.requires_grad = True
for param in model_exp2.network.encoder.backbone.blocks[-1].parameters():
    param.requires_grad = True

n = sum(p.numel() for p in model_exp2.parameters() if p.requires_grad)
print(f"Trainable parameters: {n:,}")

model_exp2.configure_optimizers = types.MethodType(configure_optimizer_exp2, model_exp2)
model_exp2.train()

trainer_exp2 = build_trainer_exp2("unfreeze-last-block", max_epochs=10)
trainer_exp2.fit(
    model=model_exp2,
    datamodule=dm_train,
    ckpt_path=find_latest_ckpt("unfreeze-last-block")
)

Trainable parameters: 7,104,788


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


epoch,▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▅▅▅▅▅▅▆▆▆▆▆██████████
losses/train_loss_cross_entropy,▂▃▂▆▃▄▇▄▃▄▃▇▆▁▂▅▃▇▆█▅▅▅▅▂▅▅▅▄▂▅█▅▃▂▄▆▄▆▁
losses/train_loss_cross_entropy_block_-1,▆▂▄▃▆▆▂▃▅▂▅▂▂█▃▃▅▃▇▇▂▄▅▄▄▄▂▅▃▃▄▃▂▄▂▄▁▃▂▂
losses/train_loss_cross_entropy_block_-2,▄▄▃▆▄█▂▃▆▄▅▅▅▃▁▃▄▂▆▅▇▆▃▃▁▂▅▄▄▅▃▂▂▅▄▂▃▆▃▅
losses/train_loss_cross_entropy_block_-3,▄▃▄▁▃█▆▃▄▄▅▄▆▁▆▂▇▇▂▄▃▄▄█▆▄▂▄▄▃▄▃▄▃▄▄▅▄▃▅
losses/train_loss_dice,▃▅▂▃▆▃▃▅▂▃▆▄▅▄▁▃█▂▄▄▅▃▂▄▃▃▄▁▄▅▂▃▆▅▅▃▆▄▅▅
losses/train_loss_dice_block_-1,▂▅▅▅▂▃▁▇▆▃▄▇▃▃▅▂▃▃▄▃▂▄▁▄▆▂▂▄▆█▆▃▄▂▃▄▃▇▃▂
losses/train_loss_dice_block_-2,▂▆▄▂▃▅▅▅▄▁▃▄▅▅▅▃▇▅█▅▃▃▆▆▅▆▅▄▂▅▄▄▅▆▅▂▃▅▆▅
losses/train_loss_dice_block_-3,▅▄▄▃▄▄█▅▃▇▅▅▅▃▅▆▅▄▅▅▅▄▂▅▅▃▃▅▅▇▃▄▄▃▂▃▃▄▂▁
losses/train_loss_mask,▃▂▅▃▃▄▁█▃▄▂▅▃▃▃▁▂▅▆▄▃▅▂▂▄▇▃▅▃▂▂▁▃▃▂▂▂▂▆▂
+10,...


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightni

No checkpoint found for 'unfreeze-last-block', starting from scratch.


...

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network   │ EoMT                   │ 93.6 M │ train │     0 │
│ 1 │ criterion │ MaskClassificationLoss │      0 │ train │     0 │
│ 2 │ metrics   │ ModuleList             │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 7.1 M                                                                                            
Non-trainable params: 86.5 M                                                                                       
Total params: 93.6 M                                                                                               
Total estimated model params size (MB): 374.302                                                                    
Modules in train mode: 304                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

INFO: mIoU: 68.4
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 68.4
INFO: mIoU: 70.4
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 70.4
INFO: mIoU: 70.9
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 70.9
INFO: mIoU: 69.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 69.5
INFO: mIoU: 72.4
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 72.4
INFO: mIoU: 73.1
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 73.1
INFO: mIoU: 73.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 73.6
INFO: mIoU: 73.3
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 73.3
INFO: mIoU: 73.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 73.5
INFO: mIoU: 73.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 73.5
INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


## 6. Experiment 3: LoRA Fine-Tuning

LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique that
avoids modifying the original weights entirely. For each target weight matrix W,
a pair of low-rank matrices B and A is introduced such that the effective weight
becomes W + BA, where rank(BA) = r << min(d, k). During training only B and A
are updated; W remains frozen.

The motivation for using LoRA here is twofold. First, it dramatically reduces the
number of trainable parameters compared to full fine-tuning while still allowing
the attention mechanism to adapt to the target domain. Second, it preserves the
pretrained weights, which is important when the downstream dataset is small
relative to the pretraining data.

We apply LoRA to the qkv projection layers of the ViT encoder, which are the most
directly involved in computing attention across patch tokens.

Reference: Hu et al. (2021). LoRA: Low-Rank Adaptation of Large Language Models.
arXiv:2106.09685.

In [ ]:
model_exp3 = build_finetune_model()

lora_config = LoraConfig(
    r=16,                    # rank of the update matrices
    lora_alpha=32,           # scaling factor (effective scale = alpha/r = 2)
    target_modules=["qkv"],  # query-key-value projections in each ViT block
    lora_dropout=0.05,
    bias="none",
)

model_exp3.network.encoder = get_peft_model(model_exp3.network.encoder, lora_config)
model_exp3.network.encoder.print_trainable_parameters()

model_exp3.configure_optimizers = types.MethodType(configure_optimizer, model_exp3)
model_exp3.train()

trainer_exp3 = build_trainer("lora-r16", max_epochs=10)
trainer_exp3.fit(
    model=model_exp3,
    datamodule=dm_train,
    ckpt_path=find_latest_ckpt("lora-r16")
)

trainable params: 589,824 || all params: 87,487,488 || trainable%: 0.6742


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


epoch,▁▁▁▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇█████
losses/train_loss_cross_entropy,▅▃▄▇▃▃▆▄█▄▄▅▂▅▄▄▃▃▃▄▂▁▄▂▅▄▄▃▄▂▃▄▃▂▂▂▂▂▂▁
losses/train_loss_cross_entropy_block_-1,▃▆▆▃▃█▅▄▅▆▄▄▄▅▅▂▃▃▆▄▄▃▂▂▄▅▄▃▅▅▃▄▂▃▃▁▄▃▂▄
losses/train_loss_cross_entropy_block_-2,▇▆▄▃▃▄▂▅▂█▄▂▂▄▆▃▃▃▂▁▅▂▄▅▃▃▂▃▂▅▂▃▃▅▃▆▂▁▂▄
losses/train_loss_cross_entropy_block_-3,█▃▄▃▃▂▂▂▂▂▂▂▂▂▁▂▁▂▂▂▁▁▁▂▁▂▂▂▁▁▂▁▁▂▂▁▂▂▂▁
losses/train_loss_dice,▆▄▃▆▅▅▅▄▅█▃▆▄▄▅▃▃▄▇▁▅▄▄▄▆▄▄▅▁▄▃▇▆▅▄▃▄▆▄▇
losses/train_loss_dice_block_-1,▄▃▆▅▁▂▆▅█▅▃▄▆▁▇▇▂▂▃▅▃▃▅▄▃▂▆▄▄▅▃▄▁▆▂▁▁▅▆▄
losses/train_loss_dice_block_-2,▃▅▅▅▂▅▃▅█▃▅▄▆▇▄▆▁▄█▄▄▅▄▅▆▅▃▄▆▃▅▇▅▄▅▅▅▇▅▄
losses/train_loss_dice_block_-3,▇▆▆▂▃▅▆▆▄▄▄▅▆▄▅▅▆▆▃▃▁▂▄▆▄▅▄▄▆▃▅▃█▇▆▄▆▂▆▁
losses/train_loss_mask,▆▅▅▄▃▅▇▅▃▇▄▄█▆▃▆▄▆▅▇▅▃▆▁▃▆▄▃▆▆▇█▅▆▄▂▄▅▅▅
+10,...


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightni

No checkpoint found for 'lora-r16', starting from scratch.


...

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ network   │ EoMT                   │ 94.2 M │ train │     0 │
│ 1 │ criterion │ MaskClassificationLoss │      0 │ train │     0 │
│ 2 │ metrics   │ ModuleList             │      0 │ train │     0 │
└───┴───────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 7.3 M                                                                                            
Non-trainable params: 86.9 M                                                                                       
Total params: 94.2 M                                                                                               
Total estimated model params size (MB): 376.661                                                                    
Modules in train mode: 426                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

INFO: mIoU: 70.2
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 70.2
INFO: mIoU: 70.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 70.5
INFO: mIoU: 72.7
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 72.7
INFO: mIoU: 74.6
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 74.6
INFO: mIoU: 74.7
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 74.7
INFO: mIoU: 74.5
INFO:lightning.pytorch.utilities.rank_zero:mIoU: 74.5


## 7. Results

The table below summarizes validation mIoU across all models evaluated on the
Cityscapes validation set. The evaluation protocol is identical in all cases:
same preprocessing, same class mapping, same iouEval evaluator as Step 4.

Fill in the values from the wandb dashboard after all runs complete.

In [ ]:
results = {
    "COCO zero-shot (Step 4)":          zeroshot_miou,
    "Head-only":                         None,   # from wandb run: head-only
    "Head + last block unfreeze":        None,   # from wandb run: unfreeze-last-block
    "LoRA (r=16)":                       None,   # from wandb run: lora-r16
    "Cityscapes pretrained (provided)":  None,   # from Step 4
}

print(f"{'Model':<45} {'val mIoU (%)':>12}")
print("-" * 59)
for name, miou in results.items():
    val = f"{miou:.2f}" if miou is not None else "pending"
    print(f"{name:<45} {val:>12}")